In [1]:
import torch

In [29]:
def get_rays(
    pa_arr: torch.Tensor, pb_arr: torch.Tensor, device: torch.device
) -> torch.Tensor:
    """
    Get rays from array of points a and array of points b
    """
    npa = pa_arr.shape[0]
    npb = pb_arr.shape[0]
    pa_arr_expanded = pa_arr.to(device).unsqueeze(1).expand((-1, npb, -1))
    pb_arr_expanded = pb_arr.to(device).unsqueeze(0).expand((npa, -1, -1))
    return torch.stack((pa_arr_expanded, pb_arr_expanded), dim=2)


def rects_edges_2d(rects):
    verts = torch.stack(
        [
            rects[:, 0],
            rects[:, 0] + rects[:, 1],
            rects[:, 0] + rects[:, 1] + rects[:, 2],
            rects[:, 0] + rects[:, 2],
        ],
        dim=1,
    )
    print(verts.shape)
    edges = torch.stack(
        [
            verts[:, (0, 1)],
            verts[:, (1, 2)],
            verts[:, (2, 3)],
            verts[:, (3, 0)],
        ],
        dim=1,
    )
    print(edges.shape)
    return edges




    # d = cross / np.linalg.norm(pb - pa_arr) * 0.5

In [20]:
rects = torch.tensor(
    [[[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[2, 1], [1.0, 0.0], [0.0, 2.0]]],
)
edges = rects_edges_2d(rects)
print(edges[1])

torch.Size([2, 4, 2])
torch.Size([2, 4, 2, 2])
tensor([[[2., 1.],
         [3., 1.]],

        [[3., 1.],
         [3., 3.]],

        [[3., 3.],
         [2., 3.]],

        [[2., 3.],
         [2., 1.]]])


In [26]:
device = torch.device("cpu")
def get_fov_centers_2d(npx_xy, mmppx_xy, device: torch.device):
    gridx, gridy = torch.meshgrid(
        torch.arange(npx_xy[0], device=device),
        torch.arange(npx_xy[1], device=device),
        indexing="ij",
    )
    grid_tensor = torch.cat((gridx, gridy), dim=-1).view(-1, 2)
    return grid_tensor * mmppx_xy.view(1, 2).expand_as(grid_tensor)

In [27]:
pa_arr = get_fov_centers_2d(torch.tensor([10, 10]), torch.tensor([1, 1]), device)

In [53]:
def abrays_cut_rects_2d(pa_arr, pb_arr, edges, device):
    n_pa = pa_arr.shape[0]
    n_pb = pb_arr.shape[0]
    n_rects = edges.shape[0]
    rays = get_rays(pa_arr, pb_arr, device)
    v1 = rays[:, :, 1] - rays[:, :, 0]
    v2 = edges[:, :, 1] - edges[:, :, 0]
    # try:
    v3 = edges[:, :, 0].view(1, -1, 4, 2).expand(n_pa, -1, -1, -1) - pa_arr.view(
        -1, 1, 1, 2
    ).expand(-1, n_rects, 4, -1)
    # except Exception as e:
    # print(edges[:, 0].view(1, -1, 4, 2).expand(n_pa, -1, -1, -1).shape)
    # print(rays.shape)
    print(v1.shape)
    print(v2.shape)
    print(v3.shape)
    # print(edges.shape)

In [54]:
pb_arr = torch.tensor([90, 5]).view(1, 2)
abrays_cut_rects_2d(pa_arr, pb_arr, edges, device)

torch.Size([100, 1, 2])
torch.Size([2, 4, 2])
torch.Size([100, 2, 4, 2])
